# Beam pre-processing experiment

## Data loading

In [1]:
import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import NETWORK_TYPE, extract_unique_npcis
from scripts.weighted_coverage import run_weighted_coverage

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt('data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

selected_campaigns = list(range(1, 10))
# Data filtering
df = filter_dataframe(
    df=df,
    operators=[10],
    include_columns=['pci', 'beam_index', 'nr_arfcn', 'operator_id', 'rsrq'],
    campaigns=selected_campaigns,
)

Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [2]:
from scripts.beamforming import get_best_beam
from scripts.utils import RF_PARAM_5G

rf_param = RF_PARAM_5G.RSRQ

control_df = df.copy()

df['best_beam'] = df['measurements_matrix'].apply(lambda x: get_best_beam(x, rf_param))

In [3]:
unique_npcis = extract_unique_npcis(df["measurements_matrix"])

_, avg_error_control, _, _ = run_weighted_coverage(
    df=control_df,
    rf_param=rf_param,
    cluster_rf_param=rf_param,
    k_max=2,
    unique_npcis=unique_npcis,
    random_seed=42,
    n_clusters=0,
    use_beam_matching=False,
)

_, avg_error, _, _ = run_weighted_coverage(
    df=df,
    rf_param=rf_param,
    cluster_rf_param=rf_param,
    k_max=2,
    unique_npcis=unique_npcis,
    random_seed=42,
    n_clusters=0,
    use_beam_matching=True,
)

print(f"""
Beam matching {avg_error.mean()}
Control {avg_error_control.mean()}
""")

avg_error


Beam matching 7.773270602217144
Control 4.1594932655771535



array([ 1.64850577,  0.75519301,  3.18007487, ...,  6.66892206,
        0.04125774, 12.70208344])